In [19]:
%pip install -q requests beautifulsoup4 lxml pandas pypdf pillow pytesseract python-dateutil

Note: you may need to restart the kernel to use updated packages.


In [20]:
import os
import re
import io
import time
import hashlib
import mimetypes
from collections import deque
from datetime import datetime
from urllib.parse import (
    urlparse, urljoin, urldefrag, parse_qsl, urlencode
)

import requests
import pandas as pd

from bs4 import BeautifulSoup
from PIL import Image, UnidentifiedImageError
from pypdf import PdfReader

try:
    import pytesseract
    OCR_AVAILABLE = True
except Exception:
    pytesseract = None
    OCR_AVAILABLE = False

print("OCR library available:", OCR_AVAILABLE)


OCR library available: True


# CONFIGURATION

In [ ]:
DATA_DIR = "portfolio_data"
CERT_IMAGE_DIR = os.path.join(DATA_DIR, "certificate_images")

MAX_INTERNAL_PAGES = 100
REQUEST_TIMEOUT = 25
REQUEST_DELAY = 0.5
MAX_TEXT_LENGTH = 120000
MAX_IMAGE_BYTES = 12 * 1024 * 1024
OCR_MAX_DIMENSION = 3000

DOCUMENT_EXTENSIONS = {
    ".pdf", ".doc", ".docx", ".txt",
    ".csv", ".xls", ".xlsx", ".ppt", ".pptx"
}

CERT_SECTION_NAMES = {
    "certificate",
    "certificates",
    "certification",
    "certifications",
    "certifications & courses",
    "certifications and courses",
    "courses & certifications",
    "courses and certifications",
    "credentials",
    "achievements and certifications"
}

SECTION_NAMES = {
    "education": {
        "education", "academic background", "academic qualifications",
        "qualifications", "educational background"
    },
    "certifications": CERT_SECTION_NAMES,
    "projects": {
        "projects", "my projects", "project work", "featured projects",
        "portfolio", "case studies", "case study", "selected projects"
    },
    "experience": {
        "experience", "work experience", "professional experience",
        "employment", "internships", "internship"
    },
    "skills": {
        "skills", "technical skills", "technologies", "tools",
        "technical expertise", "skills & technologies"
    },
    "about": {
        "about", "about me", "profile", "summary", "who i am"
    }
}

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CERT_IMAGE_DIR, exist_ok=True)

session = requests.Session()
session.headers.update({
    "User-Agent": (
        "PortfolioIQ/1.0 "
        "(portfolio analysis educational project)"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,"
        "application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8"
    ),
    "Accept-Language": "en-US,en;q=0.9",
})

print("Data directory:", os.path.abspath(DATA_DIR))
print("Certificate image directory:", os.path.abspath(CERT_IMAGE_DIR))


Data directory: /Users/varshh06/Desktop/1. PORTFOLIO INTELLIGENCE/portfolio_data
Certificate image directory: /Users/varshh06/Desktop/1. PORTFOLIO INTELLIGENCE/portfolio_data/certificate_images


In [22]:
def clean_text(value):
    if value is None:
        return ""
    value = str(value)
    value = re.sub(r"\\s+", " ", value)
    return value.strip()


def normalize_url(url):
    if not url:
        return None

    url = str(url).strip()
    url = urldefrag(url)[0]

    if not re.match(r"^https?://", url, re.I):
        url = "https://" + url

    parsed = urlparse(url)

    if parsed.scheme.lower() not in {"http", "https"}:
        return None

    domain = parsed.netloc.lower().strip()

    if domain.endswith(":80"):
        domain = domain[:-3]
    if domain.endswith(":443"):
        domain = domain[:-4]

    path = re.sub(r"/+", "/", parsed.path or "/")

    if path != "/" and path.endswith("/"):
        path = path.rstrip("/")

    useful = []
    for key, value in parse_qsl(parsed.query, keep_blank_values=True):
        key_lower = key.lower()

        if key_lower.startswith("utm_"):
            continue

        if key_lower in {"fbclid", "gclid", "ref", "source"}:
            continue

        useful.append((key, value))

    query = urlencode(useful)

    result = f"{parsed.scheme.lower()}://{domain}{path}"
    if query:
        result += "?" + query

    return result


def same_url(a, b):
    return normalize_url(a) == normalize_url(b)


def get_domain(url):
    try:
        return urlparse(url).netloc.lower().removeprefix("www.")
    except Exception:
        return ""


def get_extension(url):
    try:
        return os.path.splitext(urlparse(url).path.lower())[1]
    except Exception:
        return ""


def is_document_url(url):
    return get_extension(url) in DOCUMENT_EXTENSIONS


def is_internal_url(url, portfolio_domain):
    domain = get_domain(url)
    return bool(
        domain == portfolio_domain
        or domain.endswith("." + portfolio_domain)
    )


In [23]:
PLATFORMS = {
    "github.com": "GitHub",
    "gitlab.com": "GitLab",
    "bitbucket.org": "Bitbucket",
    "linkedin.com": "LinkedIn",
    "kaggle.com": "Kaggle",
    "leetcode.com": "LeetCode",
    "hackerrank.com": "HackerRank",
    "codeforces.com": "Codeforces",
    "codechef.com": "CodeChef",
    "geeksforgeeks.org": "GeeksforGeeks",
    "medium.com": "Medium",
    "dev.to": "Dev.to",
    "stackoverflow.com": "Stack Overflow",
    "credly.com": "Credly",
    "coursera.org": "Coursera",
    "udemy.com": "Udemy",
    "pypi.org": "PyPI",
    "npmjs.com": "NPM",
    "khanacademy.org": "Khan Academy",
    "skillshare.com": "Skillshare",
    "freecodecamp.org": "freeCodeCamp"
}


def detect_platform(url):
    if not url:
        return "Unknown"

    lower = url.lower()

    if lower.startswith("mailto:"):
        return "Email"
    if lower.startswith("tel:"):
        return "Phone"
    if is_document_url(url):
        return "Document"

    domain = get_domain(url)

    for website, platform in PLATFORMS.items():
        if domain == website or domain.endswith("." + website):
            return platform

    return "Other"


# CANDIDATE ID MANAGEMENT

In [ ]:
def get_next_candidate_id():
    path = os.path.join(DATA_DIR, "candidates.csv")

    if not os.path.exists(path):
        return "CAND_0001"

    try:
        df = pd.read_csv(path)
    except Exception:
        return "CAND_0001"

    if df.empty or "candidate_id" not in df.columns:
        return "CAND_0001"

    nums = (
        df["candidate_id"]
        .astype(str)
        .str.extract(r"CAND_(\d+)", expand=False)
    )

    nums = pd.to_numeric(nums, errors="coerce").dropna()

    next_number = int(nums.max()) + 1 if not nums.empty else 1
    return f"CAND_{next_number:04d}"


def get_or_create_candidate_id(portfolio_url):
    portfolio_url = normalize_url(portfolio_url)
    path = os.path.join(DATA_DIR, "candidates.csv")

    if os.path.exists(path):
        try:
            df = pd.read_csv(path)

            if not df.empty and "portfolio_url" in df.columns:
                normalized_existing = (
                    df["portfolio_url"]
                    .astype(str)
                    .map(normalize_url)
                )

                matches = df[normalized_existing == portfolio_url]

                if not matches.empty:
                    candidate_id = str(matches.iloc[0]["candidate_id"])
                    print(f"Existing portfolio found -> {candidate_id}")
                    return candidate_id

        except Exception as exc:
            print("Warning while checking candidates.csv:", exc)

    candidate_id = get_next_candidate_id()
    print(f"New portfolio -> {candidate_id}")
    return candidate_id


In [ ]:
portfolio_url = input("Enter candidate portfolio URL: ").strip()
portfolio_url = normalize_url(portfolio_url)

if not portfolio_url:
    raise ValueError("Invalid portfolio URL.")

portfolio_domain = get_domain(portfolio_url)
candidate_id = get_or_create_candidate_id(portfolio_url)

print("Portfolio:", portfolio_url)
print("Domain:", portfolio_domain)
print("Candidate ID:", candidate_id)


Existing portfolio found -> CAND_0001
Portfolio: https://varshh-hub.github.io/VARSHA---portfolio
Domain: varshh-hub.github.io
Candidate ID: CAND_0001


In [26]:
def download_url(url):
    try:
        response = session.get(
            url,
            timeout=REQUEST_TIMEOUT,
            allow_redirects=True
        )

        return {
            "requested_url": url,
            "final_url": response.url,
            "status_code": response.status_code,
            "content_type": response.headers.get("Content-Type", ""),
            "content": response.content,
            "error": None
        }

    except Exception as exc:
        return {
            "requested_url": url,
            "final_url": None,
            "status_code": None,
            "content_type": "",
            "content": None,
            "error": str(exc)
        }


def extract_links(soup, current_url, portfolio_domain):
    internal = set()
    external = set()
    documents = set()
    emails = set()
    phones = set()

    for tag in soup.find_all("a", href=True):
        href = tag.get("href", "").strip()

        if not href:
            continue

        lower = href.lower()

        if lower.startswith(("javascript:", "#", "data:")):
            continue

        if lower.startswith("mailto:"):
            emails.add(href)
            continue

        if lower.startswith("tel:"):
            phones.add(href)
            continue

        absolute = normalize_url(urljoin(current_url, href))

        if not absolute:
            continue

        if is_document_url(absolute):
            documents.add(absolute)
        elif is_internal_url(absolute, portfolio_domain):
            internal.add(absolute)
        else:
            external.add(absolute)

    return {
        "internal": sorted(internal),
        "external": sorted(external),
        "documents": sorted(documents),
        "emails": sorted(emails),
        "phones": sorted(phones)
    }


In [ ]:
def parse_html(html, current_url, portfolio_domain):
    soup = BeautifulSoup(html, "lxml")

    title = ""
    if soup.title:
        title = clean_text(soup.title.get_text(" ", strip=True))

    meta_description = ""
    meta = soup.find(
        "meta",
        attrs={"name": re.compile(r"^description$", re.I)}
    )
    if meta:
        meta_description = clean_text(meta.get("content", ""))

    for tag in soup.find_all(["script", "style", "noscript", "template"]):
        tag.decompose()

    headings = []
    for tag in soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"]):
        value = clean_text(tag.get_text(" ", strip=True))
        if value:
            headings.append({
                "tag": tag.name,
                "text": value
            })

    paragraphs = [
        clean_text(tag.get_text(" ", strip=True))
        for tag in soup.find_all("p")
        if clean_text(tag.get_text(" ", strip=True))
    ]

    list_items = [
        clean_text(tag.get_text(" ", strip=True))
        for tag in soup.find_all("li")
        if clean_text(tag.get_text(" ", strip=True))
    ]

    images = []

    for img in soup.find_all("img"):
        src = (
            img.get("src")
            or img.get("data-src")
            or img.get("data-lazy-src")
            or img.get("data-original")
        )

        srcset = img.get("srcset") or img.get("data-srcset")

        if not src and srcset:
            src = srcset.split(",")[0].strip().split(" ")[0]

        if not src:
            continue

        image_url = urljoin(current_url, src)

        images.append({
            "url": image_url,
            "alt": clean_text(img.get("alt", "")),
            "title": clean_text(img.get("title", "")),
            "class": " ".join(img.get("class", [])),
            "id": clean_text(img.get("id", "")),
            "parent_text": clean_text(
                img.parent.get_text(" ", strip=True)
                if img.parent else ""
            )
        })

    links = extract_links(
        soup,
        current_url,
        portfolio_domain
    )

    text = soup.get_text(separator="\n", strip=True)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = text[:MAX_TEXT_LENGTH]

    return {
        "soup": soup,
        "title": title,
        "meta_description": meta_description,
        "headings": headings,
        "paragraphs": paragraphs,
        "list_items": list_items,
        "images": images,
        "text": text,
        "links": links
    }


# SECTION DETECTION

In [ ]:
def canonical_section_name(text):
    value = clean_text(text).lower()
    value = re.sub(r"[:|•·]+$", "", value).strip()

    for section, names in SECTION_NAMES.items():
        if value in names:
            return section

    return None


def find_section_blocks(soup):
    blocks = {
        "education": [],
        "certifications": [],
        "projects": [],
        "experience": [],
        "skills": [],
        "about": []
    }

    headings = soup.find_all(["h1", "h2", "h3", "h4"])

    for heading in headings:
        section = canonical_section_name(
            heading.get_text(" ", strip=True)
        )

        if not section:
            continue

        collected = []
        level = int(heading.name[1])

        for sibling in heading.next_siblings:
            if getattr(sibling, "name", None):
                sibling_name = sibling.name

                if (
                    sibling_name in {"h1", "h2", "h3", "h4"}
                    and int(sibling_name[1]) <= level
                ):
                    break

                collected.append(sibling)

        if not collected and heading.parent:
            collected = [heading.parent]

        blocks[section].append({
            "heading": clean_text(heading.get_text(" ", strip=True)),
            "nodes": collected,
            "heading_tag": heading.name
        })

    return blocks


def block_text(block):
    values = []

    for node in block.get("nodes", []):
        try:
            value = clean_text(node.get_text("\n", strip=True))
        except Exception:
            value = ""

        if value:
            values.append(value)

    return "\n".join(values)


def block_images(block, current_url):
    found = []

    for node in block.get("nodes", []):
        if not getattr(node, "find_all", None):
            continue

        for img in node.find_all("img"):
            src = (
                img.get("src")
                or img.get("data-src")
                or img.get("data-lazy-src")
                or img.get("data-original")
            )

            srcset = img.get("srcset") or img.get("data-srcset")

            if not src and srcset:
                src = srcset.split(",")[0].strip().split(" ")[0]

            if not src:
                continue

            found.append({
                "url": urljoin(current_url, src),
                "alt": clean_text(img.get("alt", "")),
                "title": clean_text(img.get("title", "")),
                "class": " ".join(img.get("class", [])),
                "id": clean_text(img.get("id", "")),
                "parent_text": clean_text(
                    img.parent.get_text(" ", strip=True)
                    if img.parent else ""
                )
            })

    return found


# SKILLS

In [ ]:
SKILL_ALIASES = {
    "python": ["python"],
    "java": ["java"],
    "javascript": ["javascript", "js"],
    "typescript": ["typescript"],
    "c++": ["c++", "cpp"],
    "c": ["c programming"],
    "c#": ["c#", "c sharp"],
    "sql": ["sql", "structured query language"],
    "excel": ["microsoft excel", "ms excel", "excel"],
    "pandas": ["pandas"],
    "numpy": ["numpy"],
    "scikit-learn": ["scikit-learn", "sklearn", "scikit learn"],
    "tensorflow": ["tensorflow"],
    "pytorch": ["pytorch"],
    "xgboost": ["xgboost"],
    "machine learning": ["machine learning"],
    "deep learning": ["deep learning"],
    "statistics": ["statistics", "statistical analysis"],
    "power bi": ["power bi", "powerbi"],
    "tableau": ["tableau"],
    "html": ["html"],
    "css": ["css"],
    "bootstrap": ["bootstrap"],
    "react": ["react", "react.js", "reactjs"],
    "node.js": ["node.js", "nodejs"],
    "flask": ["flask"],
    "fastapi": ["fastapi"],
    "django": ["django"],
    "mysql": ["mysql"],
    "postgresql": ["postgresql", "postgres"],
    "mongodb": ["mongodb", "mongo db"],
    "aws": ["aws", "amazon web services"],
    "azure": ["azure", "microsoft azure"],
    "gcp": ["gcp", "google cloud", "google cloud platform"],
    "docker": ["docker"],
    "git": ["git"],
    "github": ["github"],
    "jupyter": ["jupyter", "jupyter notebook", "jupyter notebooks"],
    "selenium": ["selenium"],
    "beautifulsoup": ["beautifulsoup", "beautiful soup"],
    "pyspark": ["pyspark", "spark"],
    "opencv": ["opencv", "open cv"],
    "nlp": ["nlp", "natural language processing"],
    "generative ai": ["generative ai", "genai"],
    "streamlit": ["streamlit"],
    "powerpoint": ["powerpoint", "microsoft powerpoint"]
}


def extract_skills(text):
    if not text:
        return []

    found = []

    for canonical, aliases in SKILL_ALIASES.items():
        for alias in aliases:
            pattern = r"(?<!\w)" + re.escape(alias.lower()) + r"(?!\w)"
            if re.search(pattern, text.lower()):
                found.append(canonical)
                break

    return sorted(set(found))


def extract_skill_evidence(text, source_url, source_type, candidate_id):
    records = []

    if not text:
        return records

    chunks = [
        clean_text(x)
        for x in re.split(r"\n+|(?<=[.!?])\s+", text)
        if clean_text(x)
    ]

    for chunk in chunks:
        for skill in extract_skills(chunk):
            records.append({
                "candidate_id": candidate_id,
                "skill": skill,
                "evidence_text": chunk,
                "source_url": source_url,
                "source_type": source_type
            })

    return records


# IDENTITY / ROLE / GOAL

In [ ]:
ROLE_PATTERNS = [
    r"\baspiring\s+(?:to become\s+)?([^.!|\n]{3,80})",
    r"\bwannabe\s+([^.!|\n]{3,80})",
    r"\bseeking\s+(?:a\s+)?(?:role\s+as\s+)?([^.!|\n]{3,80})",
    r"\blooking for\s+(?:a\s+)?(?:role\s+as\s+)?([^.!|\n]{3,80})",
    r"\bcareer goal\s*[:\-]\s*([^.!|\n]{3,80})",
    r"\bgoal\s*[:\-]\s*([^.!|\n]{3,80})",
    r"\bwant to become\s+(?:a\s+)?([^.!|\n]{3,80})",
    r"\bwant to work as\s+(?:a\s+)?([^.!|\n]{3,80})"
]


def extract_name(soup, title=""):
    candidates = []

    for tag in soup.find_all(["h1", "h2"]):
        value = clean_text(tag.get_text(" ", strip=True))
        if value:
            candidates.append(value)

    for value in candidates:
        lower = value.lower()
        if any(x in lower for x in ["portfolio", "data science portfolio", "welcome"]):
            continue
        if 1 < len(value.split()) <= 5 and len(value) <= 70:
            return value

    title_clean = clean_text(title)
    if "|" in title_clean:
        left = clean_text(title_clean.split("|")[0])
        if left and len(left.split()) <= 5:
            return left

    return ""


def extract_role_goal(soup, page_text, title=""):
    evidence = []

    for tag in soup.find_all(["h1", "h2", "h3", "p", "li"]):
        text = clean_text(tag.get_text(" ", strip=True))
        if text:
            evidence.append(text)

    combined = "\n".join(evidence + [clean_text(title), clean_text(page_text[:5000])])

    for pattern in ROLE_PATTERNS:
        match = re.search(pattern, combined, re.I)
        if match:
            role = clean_text(match.group(1))
            role = re.split(r"[.;|]", role)[0].strip()
            if role:
                return role

    # Title fallback.
    title_lower = title.lower()
    role_keywords = [
        "data scientist", "data science", "machine learning engineer",
        "ml engineer", "ai engineer", "software engineer",
        "frontend developer", "backend developer", "full stack developer",
        "data analyst", "business analyst", "web developer"
    ]

    for keyword in role_keywords:
        if keyword in title_lower:
            return keyword.title()

    return ""


# EDUCATION EXTRACTION

In [ ]:
DEGREE_PATTERNS = [
    r"\bB\.?\s*Sc\.?\b",
    r"\bB\.?\s*Tech\.?\b",
    r"\bB\.?\s*E\.?\b",
    r"\bB\.?\s*CA\.?\b",
    r"\bB\.?\s*Com\.?\b",
    r"\bB\.?\s*A\.?\b",
    r"\bM\.?\s*Sc\.?\b",
    r"\bM\.?\s*Tech\.?\b",
    r"\bM\.?\s*E\.?\b",
    r"\bM\.?\s*CA\.?\b",
    r"\bM\.?\s*Com\.?\b",
    r"\bM\.?\s*A\.?\b",
    r"\bMBA\b",
    r"\bBBA\b",
    r"\bBCA\b",
    r"\bMCA\b",
    r"\bBE\b",
    r"\bME\b",
    r"\bBachelor(?:'s)?\b",
    r"\bMaster(?:'s)?\b",
    r"\bDiploma\b",
    r"\bPh\.?\s*D\.?\b",
    r"\bPhD\b",
    r"\bUndergraduate\b",
    r"\bPostgraduate\b",
    r"\bHigher Secondary\b",
    r"\bHigh School\b",
    r"\bSecondary School\b"
]

FIELD_KEYWORDS = [
    "artificial intelligence and machine learning",
    "artificial intelligence & machine learning",
    "artificial intelligence",
    "machine learning",
    "data science",
    "computer science",
    "information technology",
    "information systems",
    "computer applications",
    "business administration",
    "electronics and communication",
    "electronics",
    "electrical engineering",
    "mechanical engineering",
    "civil engineering",
    "commerce",
    "statistics",
    "mathematics",
    "physics",
    "chemistry"
]

INSTITUTION_KEYWORDS = [
    "college",
    "university",
    "institute",
    "school",
    "academy",
    "polytechnic",
    "vidyalaya"
]


def extract_years(text):
    return sorted(
        set(
            re.findall(
                r"\b(?:19|20)\d{2}\b",
                text or ""
            )
        )
    )


def extract_degree(text):
    text = clean_text(text)

    for pattern in sorted(DEGREE_PATTERNS, key=len, reverse=True):
        match = re.search(pattern, text, re.I)

        if match:
            return clean_text(match.group(0))

    return ""


def extract_field_of_study(text):
    text = clean_text(text)
    lower = text.lower()

    # First look for known fields.
    for field in sorted(FIELD_KEYWORDS, key=len, reverse=True):
        if field.lower() in lower:
            return field.title()

    patterns = [
        r"\bin\s+([A-Za-z][A-Za-z &/,-]{2,100})",
        r"\bmajor\s*[:\-]\s*([A-Za-z][A-Za-z &/,-]{2,100})",
        r"\bspecialization\s*[:\-]\s*([A-Za-z][A-Za-z &/,-]{2,100})",
        r"\bspecialisation\s*[:\-]\s*([A-Za-z][A-Za-z &/,-]{2,100})",
        r"\bfield\s*[:\-]\s*([A-Za-z][A-Za-z &/,-]{2,100})"
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.I)

        if match:
            value = clean_text(match.group(1))

            value = re.split(
                r"\b(?:from|at|during|between)\b",
                value,
                flags=re.I
            )[0]

            value = re.sub(
                r"\b(?:19|20)\d{2}\b.*$",
                "",
                value
            ).strip(" -,:|")

            if 3 <= len(value) <= 100:
                return value

    return ""


def extract_institution(text):
    text = clean_text(text)

    patterns = [
        r"(?:institution|college|university|school|institute)"
        r"\s*[:\-]\s*([^|]+)",

        r"(?:studied at|studying at|graduated from|"
        r"educated at|attended|completed at)\s+"
        r"([^|]+)",

        r"\bfrom\s+([^|]+)",

        r"\bat\s+([^|]+)"
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.I)

        if match:
            value = clean_text(match.group(1))

            value = re.split(
                r"\b(?:19|20)\d{2}\b",
                value
            )[0]

            value = value.strip(" -,:|.")

            if any(
                keyword in value.lower()
                for keyword in INSTITUTION_KEYWORDS
            ):
                return value

    pattern = (
        r"\b([A-Z][A-Za-z0-9&.'’\- ]{2,150}"
        r"(?:College|University|Institute|School|Academy|"
        r"Polytechnic|Vidyalaya))\b"
    )

    match = re.search(pattern, text)

    if match:
        return clean_text(match.group(1))

    for keyword in INSTITUTION_KEYWORDS:

        pattern = (
            r"([A-Za-z0-9&.'’\- ]{3,150}"
            + re.escape(keyword)
            + r"(?:[A-Za-z0-9&.'’\- ]{0,100}))"
        )

        match = re.search(pattern, text, re.I)

        if match:
            value = clean_text(match.group(1))

            if 3 <= len(value) <= 180:
                return value

    return ""


def get_education_containers(soup):
    containers = []

    for tag in soup.find_all(
        ["section", "div", "article", "main", "aside"]
    ):

        class_name = " ".join(
            tag.get("class", [])
        )

        tag_id = tag.get("id", "")

        marker = (
            class_name + " " + tag_id
        ).lower()

        if any(
            word in marker
            for word in [
                "education",
                "educational",
                "academic",
                "qualification"
            ]
        ):

            containers.append(tag)

    for heading in soup.find_all(
        ["h1", "h2", "h3", "h4", "h5", "h6"]
    ):

        heading_text = clean_text(
            heading.get_text(" ", strip=True)
        ).lower()

        if heading_text in {
            "education",
            "educational background",
            "academic background",
            "academic qualifications",
            "qualifications",
            "education & qualifications",
            "education and qualifications"
        }:

            # Parent section.
            if heading.parent:
                containers.append(
                    heading.parent
                )

            sibling_text = []

            for sibling in heading.next_siblings:

                if getattr(sibling, "name", None) in {
                    "h1",
                    "h2",
                    "h3",
                    "h4",
                    "h5",
                    "h6"
                }:
                    break

                if hasattr(sibling, "get_text"):
                    value = clean_text(
                        sibling.get_text(
                            " ",
                            strip=True
                        )
                    )

                    if value:
                        sibling_text.append(value)

            if sibling_text:
                containers.append(
                    " | ".join(sibling_text)
                )

    return containers


def get_education_items(container):
    """
    Extract individual education cards/items instead of treating
    the entire education section as one record.
    """

    if isinstance(container, str):
        return [container]

    items = []

    for tag in container.find_all(
        ["article", "li"],
        recursive=True
    ):

        text = clean_text(
            tag.get_text(" ", strip=True)
        )

        if 15 <= len(text) <= 2000:
            items.append(text)

    for tag in container.find_all(
        "div",
        recursive=True
    ):

        class_name = " ".join(
            tag.get("class", [])
        ).lower()

        tag_id = tag.get("id", "").lower()

        marker = class_name + " " + tag_id

        if any(
            word in marker
            for word in [
                "education",
                "degree",
                "academic",
                "qualification",
                "college",
                "university"
            ]
        ):

            text = clean_text(
                tag.get_text(" ", strip=True)
            )

            if 15 <= len(text) <= 2000:
                items.append(text)

    unique = []

    for item in items:

        duplicate = False

        for existing in unique:

            if item.lower() == existing.lower():
                duplicate = True
                break

            if (
                len(item) > len(existing)
                and existing.lower() in item.lower()
            ):
                duplicate = True
                break

        if not duplicate:
            unique.append(item)

    if not unique:

        text = clean_text(
            container.get_text(" ", strip=True)
        )

        if text:
            unique = [text]

    return unique


def extract_education(page, candidate_id):

    soup = page["soup"]

    records = []

    containers = get_education_containers(soup)
    for container in containers:

        items = get_education_items(container)

        for item in items:

            item = clean_text(item)

            if not item:
                continue

            degree = extract_degree(item)

            institution = extract_institution(item)

            field = extract_field_of_study(item)

            years = extract_years(item)

            start_year = ""
            end_year = ""

            if len(years) >= 2:

                start_year = years[0]
                end_year = years[-1]

            elif len(years) == 1:

                end_year = years[0]

            if not any([
                degree,
                institution,
                field,
                years
            ]):
                continue

            records.append({

                "candidate_id": candidate_id,

                "qualification": degree,

                "institution": institution,

                "field_of_study": field,

                "start_year": start_year,

                "end_year": end_year,

                "year": end_year or start_year,

                "source_url": page["url"],

                "evidence_text": item,

                "source_type": "education"
            })

    if not records:

        lines = [
            clean_text(line)
            for line in page.get(
                "text",
                ""
            ).splitlines()
            if clean_text(line)
        ]

        for i, line in enumerate(lines):

            lower = line.lower()

            education_signal = (
                extract_degree(line)
                or extract_institution(line)
                or any(
                    word in lower
                    for word in [
                        "b.sc",
                        "bsc",
                        "b.tech",
                        "btech",
                        "m.sc",
                        "msc",
                        "m.tech",
                        "mtech",
                        "bachelor",
                        "master",
                        "college",
                        "university",
                        "institute",
                        "school",
                        "education",
                        "graduated",
                        "studied"
                    ]
                )
            )

            if not education_signal:
                continue

            context = lines[
                max(0, i - 1):
                min(len(lines), i + 3)
            ]

            evidence = clean_text(
                " | ".join(context)
            )

            degree = extract_degree(evidence)

            institution = extract_institution(
                evidence
            )

            field = extract_field_of_study(
                evidence
            )

            years = extract_years(
                evidence
            )

            if not any([
                degree,
                institution,
                field,
                years
            ]):
                continue

            start_year = ""
            end_year = ""

            if len(years) >= 2:

                start_year = years[0]
                end_year = years[-1]

            elif len(years) == 1:

                end_year = years[0]

            records.append({

                "candidate_id": candidate_id,

                "qualification": degree,

                "institution": institution,

                "field_of_study": field,

                "start_year": start_year,

                "end_year": end_year,

                "year": end_year or start_year,

                "source_url": page["url"],

                "evidence_text": evidence,

                "source_type": "education_fallback"
            })

    unique_records = []

    seen = set()

    for record in records:

        key = (
            record["qualification"].lower(),
            record["institution"].lower(),
            record["field_of_study"].lower(),
            record["start_year"],
            record["end_year"]
        )

        if key not in seen:

            seen.add(key)

            unique_records.append(record)

    return unique_records

# PROJECTS

In [ ]:
PROJECT_NAME_KEYS = {
    "project", "project-name", "project_name",
    "title", "name", "card-title"
}


def extract_projects(page, candidate_id):
    soup = page["soup"]
    blocks = find_section_blocks(soup)
    records = []

    for block in blocks["projects"]:
        section_nodes = block.get("nodes", [])

        for node in section_nodes:
            if not getattr(node, "find_all", None):
                continue

            # Prefer cards/articles/items inside the project section.
            candidates = node.find_all(
                ["article", "li", "div"],
                recursive=True
            )

            for card in candidates:
                text = clean_text(card.get_text(" ", strip=True))

                if len(text) < 20 or len(text) > 2000:
                    continue

                heading = card.find(["h2", "h3", "h4", "h5"])
                name = clean_text(
                    heading.get_text(" ", strip=True)
                    if heading else ""
                )

                if not name:
                    for attr in ["data-project", "data-title", "id", "class"]:
                        value = card.get(attr)
                        if isinstance(value, list):
                            value = " ".join(value)
                        value = clean_text(value)
                        if value and value.lower() not in {
                            "card", "project", "project-card"
                        }:
                            name = value
                            break

                if not name:
                    # First short sentence as a fallback title.
                    pieces = re.split(r"[.!?]", text)
                    name = clean_text(pieces[0])[:120]

                links = []
                for a in card.find_all("a", href=True):
                    link = normalize_url(urljoin(page["url"], a["href"]))
                    if link:
                        links.append(link)

                skills = extract_skills(text)

                records.append({
                    "candidate_id": candidate_id,
                    "project_name": name,
                    "description": text,
                    "skills": ", ".join(skills),
                    "project_url": links[0] if links else "",
                    "source_url": page["url"],
                    "evidence_text": text,
                    "source_type": "project"
                })

    if not records:
        for heading in soup.find_all(["h2", "h3", "h4"]):
            value = clean_text(heading.get_text(" ", strip=True))
            if not value or canonical_section_name(value):
                continue

            lower = value.lower()
            if any(k in lower for k in [
                "project", "prediction", "dashboard", "website",
                "application", "analyzer", "management", "system"
            ]):
                parent = heading.parent
                evidence = clean_text(
                    parent.get_text(" ", strip=True)
                    if parent else value
                )

                records.append({
                    "candidate_id": candidate_id,
                    "project_name": value,
                    "description": evidence,
                    "skills": ", ".join(extract_skills(evidence)),
                    "project_url": "",
                    "source_url": page["url"],
                    "evidence_text": evidence,
                    "source_type": "project"
                })

    return records


# EXPERIENCE

In [ ]:
EXPERIENCE_ROLE_WORDS = [
    "developer", "engineer", "analyst", "scientist",
    "designer", "manager", "consultant", "intern",
    "trainee", "associate", "researcher", "specialist"
]


def extract_experience(page, candidate_id):
    soup = page["soup"]
    blocks = find_section_blocks(soup)
    records = []

    for block in blocks["experience"]:
        for node in block.get("nodes", []):
            if not getattr(node, "find_all", None):
                continue

            candidates = node.find_all(
                ["article", "li", "div"],
                recursive=True
            )

            for card in candidates:
                text = clean_text(card.get_text(" ", strip=True))

                if len(text) < 15 or len(text) > 1800:
                    continue

                heading = card.find(["h2", "h3", "h4", "h5"])
                role = clean_text(
                    heading.get_text(" ", strip=True)
                    if heading else ""
                )

                if not role:
                    for line in re.split(r"[\n|]", text):
                        line = clean_text(line)
                        if any(word in line.lower() for word in EXPERIENCE_ROLE_WORDS):
                            role = line
                            break

                if not role:
                    continue

                records.append({
                    "candidate_id": candidate_id,
                    "role": role,
                    "company": "",
                    "description": text,
                    "skills": ", ".join(extract_skills(text)),
                    "source_url": page["url"],
                    "evidence_text": text,
                    "source_type": "experience"
                })

    return records


# CERTIFICATE IMAGE DETECTION + OCR

In [ ]:
CERT_KEYWORDS = {
    "certificate", "certification", "certified",
    "credential", "credentials", "course completion",
    "certificate of completion", "achievement",
    "coursera", "udemy", "linkedin learning",
    "nptel", "great learning", "simplilearn",
    "edx", "ibm", "microsoft", "google", "aws"
}


def certificate_likelihood(image_record):
    text = " ".join([
        image_record.get("alt", ""),
        image_record.get("title", ""),
        image_record.get("class", ""),
        image_record.get("id", ""),
        image_record.get("parent_text", ""),
        image_record.get("url", "")
    ]).lower()

    score = 0

    for keyword in CERT_KEYWORDS:
        if keyword in text:
            score += 2

    if re.search(r"(cert|certificate|credential|course|badge)", text, re.I):
        score += 3

    return score


def download_image(image_url):
    try:
        response = session.get(
            image_url,
            timeout=REQUEST_TIMEOUT,
            allow_redirects=True
        )

        if response.status_code >= 400:
            return None, f"HTTP {response.status_code}"

        content_type = response.headers.get("Content-Type", "").lower()

        if len(response.content) > MAX_IMAGE_BYTES:
            return None, "Image exceeds MAX_IMAGE_BYTES"

        if not (
            content_type.startswith("image/")
            or image_url.lower().split("?")[0].endswith(
                (".jpg", ".jpeg", ".png", ".webp", ".gif", ".bmp")
            )
        ):
            return None, "URL did not return an image"

        return response.content, None

    except Exception as exc:
        return None, str(exc)


def image_hash(content):
    return hashlib.sha256(content).hexdigest()


def save_image(content, image_url):
    digest = image_hash(content)

    try:
        image = Image.open(io.BytesIO(content))
        extension = (
            image.format.lower()
            if image.format
            else "jpg"
        )

        if extension == "jpeg":
            extension = "jpg"

    except Exception:
        extension = "jpg"

    filename = f"{digest[:24]}.{extension}"
    path = os.path.join(CERT_IMAGE_DIR, filename)

    if not os.path.exists(path):
        with open(path, "wb") as handle:
            handle.write(content)

    return path, digest


def ocr_image(content):
    if not OCR_AVAILABLE:
        return ""

    try:
        image = Image.open(io.BytesIO(content))

        image.thumbnail(
            (OCR_MAX_DIMENSION, OCR_MAX_DIMENSION)
        )

        text = pytesseract.image_to_string(image)

        return clean_text(text)

    except (UnidentifiedImageError, Exception):
        return ""


def infer_certificate_name(ocr_text, image_record):
    if ocr_text:
        lines = [
            clean_text(x)
            for x in ocr_text.splitlines()
            if clean_text(x)
        ]

        for line in lines:
            lower = line.lower()

            if any(k in lower for k in [
                "certificate", "certification", "course",
                "completion", "professional", "learning"
            ]):
                if 3 <= len(line) <= 180:
                    return line

        # Otherwise use the longest useful early line.
        for line in lines[:8]:
            if 8 <= len(line) <= 180:
                return line

    alt = clean_text(image_record.get("alt", ""))
    if alt:
        return alt

    title = clean_text(image_record.get("title", ""))
    if title:
        return title

    return ""

def extract_year(text):
    """
    Extract the most likely year from OCR/text.
    Returns the first 4-digit year between 1900 and 2099.
    """
    if not text:
        return ""

    years = re.findall(r"\b(?:19|20)\d{2}\b", str(text))

    if years:
        return years[0]

    return ""

def extract_certificate_images(page, candidate_id):
    soup = page["soup"]
    blocks = find_section_blocks(soup)

    all_images = []

    for block in blocks["certifications"]:
        all_images.extend(
            block_images(block, page["url"])
        )

    if not all_images:
        for img in page.get("images", []):
            if certificate_likelihood(img) >= 3:
                all_images.append(img)

    unique = {}
    for img in all_images:
        unique[normalize_url(img["url"])] = img

    records = []

    for image_url, img in unique.items():
        if not image_url:
            continue

        content, error = download_image(image_url)

        record = {
            "candidate_id": candidate_id,
            "certificate_name": "",
            "issuer": "",
            "date": "",
            "image_url": image_url,
            "local_image_path": "",
            "image_sha256": "",
            "ocr_text": "",
            "source_url": page["url"],
            "source_type": "certificate_image",
            "error": error or ""
        }

        if content is None:
            records.append(record)
            continue

        local_path, digest = save_image(
            content,
            image_url
        )

        ocr_text = ocr_image(content)

        record["local_image_path"] = local_path
        record["image_sha256"] = digest
        record["ocr_text"] = ocr_text
        record["certificate_name"] = infer_certificate_name(
            ocr_text,
            img
        )

        year = extract_year(ocr_text)
        record["date"] = year

        issuer_candidates = [
            "Coursera", "Udemy", "NPTEL", "LinkedIn Learning",
            "Microsoft", "Google", "IBM", "AWS", "Great Learning",
            "Simplilearn", "edX", "freeCodeCamp"
        ]

        for issuer in issuer_candidates:
            if issuer.lower() in ocr_text.lower():
                record["issuer"] = issuer
                break

        records.append(record)

    return records


# CRAWLER

In [ ]:
def crawl_portfolio(start_url):
    start_url = normalize_url(start_url)
    domain = get_domain(start_url)

    queue = deque([start_url])
    visited = set()

    pages = []
    external_links = set()
    document_links = set()
    email_links = set()
    phone_links = set()

    while queue and len(visited) < MAX_INTERNAL_PAGES:
        current_url = queue.popleft()

        if current_url in visited:
            continue

        visited.add(current_url)

        print(
            f"[{len(visited)}/{MAX_INTERNAL_PAGES}] "
            f"{current_url}"
        )

        result = download_url(current_url)

        if result["content"] is None:
            pages.append({
                "url": current_url,
                "final_url": result["final_url"],
                "status_code": result["status_code"],
                "content_type": result["content_type"],
                "title": "",
                "meta_description": "",
                "headings": [],
                "paragraphs": [],
                "list_items": [],
                "images": [],
                "text": "",
                "soup": BeautifulSoup("", "lxml"),
                "links": {
                    "internal": [],
                    "external": [],
                    "documents": [],
                    "emails": [],
                    "phones": []
                },
                "error": result["error"]
            })
            continue

        content_type = (
            result["content_type"] or ""
        ).lower()

        if "text/html" not in content_type:
            continue

        parsed = parse_html(
            result["content"],
            result["final_url"] or current_url,
            domain
        )

        links = parsed["links"]

        external_links.update(links["external"])
        document_links.update(links["documents"])
        email_links.update(links["emails"])
        phone_links.update(links["phones"])

        page = {
            "url": current_url,
            "final_url": result["final_url"],
            "status_code": result["status_code"],
            "content_type": result["content_type"],
            "title": parsed["title"],
            "meta_description": parsed["meta_description"],
            "headings": parsed["headings"],
            "paragraphs": parsed["paragraphs"],
            "list_items": parsed["list_items"],
            "images": parsed["images"],
            "text": parsed["text"],
            "soup": parsed["soup"],
            "links": links,
            "error": None
        }

        pages.append(page)

        for link in links["internal"]:
            if link not in visited:
                queue.append(link)

        time.sleep(REQUEST_DELAY)

    return {
        "pages": pages,
        "external_links": sorted(external_links),
        "document_links": sorted(document_links),
        "email_links": sorted(email_links),
        "phone_links": sorted(phone_links)
    }


crawl_result = crawl_portfolio(portfolio_url)

print("\nCRAWLING COMPLETE")
print("Pages:", len(crawl_result["pages"]))
print("External links:", len(crawl_result["external_links"]))
print("Documents:", len(crawl_result["document_links"]))
print("Email links:", len(crawl_result["email_links"]))
print("Phone links:", len(crawl_result["phone_links"]))


[1/100] https://varshh-hub.github.io/VARSHA---portfolio

CRAWLING COMPLETE
Pages: 1
External links: 2
Documents: 1
Email links: 1
Phone links: 1


# BUILD STRUCTURED RECORDS

In [ ]:
pages = crawl_result["pages"]

page_records = []
skill_records = []
project_records = []
experience_records = []
education_records = []
certificate_records = []
link_records = []

for page in pages:
    source_url = page["url"]

    page_records.append({
        "candidate_id": candidate_id,
        "source_url": source_url,
        "final_url": page.get("final_url", ""),
        "title": page.get("title", ""),
        "meta_description": page.get("meta_description", ""),
        "evidence_text": page.get("text", ""),
        "status_code": page.get("status_code"),
        "content_type": page.get("content_type", ""),
        "error": page.get("error"),
        "scraped_at": datetime.now().isoformat()
    })

    skill_records.extend(
        extract_skill_evidence(
            page.get("text", ""),
            source_url,
            "portfolio_page",
            candidate_id
        )
    )

    project_records.extend(
        extract_projects(page, candidate_id)
    )

    experience_records.extend(
        extract_experience(page, candidate_id)
    )

    education_records.extend(
        extract_education(page, candidate_id)
    )

    certificate_records.extend(
        extract_certificate_images(page, candidate_id)
    )

    for link in page["links"]["internal"]:
        link_records.append({
            "candidate_id": candidate_id,
            "source_url": source_url,
            "link_url": link,
            "link_type": "internal",
            "platform": "Portfolio",
            "is_internal": True,
            "status": "discovered"
        })

    for link in page["links"]["external"]:
        link_records.append({
            "candidate_id": candidate_id,
            "source_url": source_url,
            "link_url": link,
            "link_type": "external",
            "platform": detect_platform(link),
            "is_internal": False,
            "status": "discovered"
        })

    for link in page["links"]["documents"]:
        link_records.append({
            "candidate_id": candidate_id,
            "source_url": source_url,
            "link_url": link,
            "link_type": "document",
            "platform": "Document",
            "is_internal": is_internal_url(link, portfolio_domain),
            "status": "discovered"
        })

for email in crawl_result["email_links"]:
    link_records.append({
        "candidate_id": candidate_id,
        "source_url": portfolio_url,
        "link_url": email,
        "link_type": "email",
        "platform": "Email",
        "is_internal": False,
        "status": "discovered"
    })

for phone in crawl_result["phone_links"]:
    link_records.append({
        "candidate_id": candidate_id,
        "source_url": portfolio_url,
        "link_url": phone,
        "link_type": "phone",
        "platform": "Phone",
        "is_internal": False,
        "status": "discovered"
    })


pages_df = pd.DataFrame(page_records)
skills_df = pd.DataFrame(skill_records)
projects_df = pd.DataFrame(project_records)
experience_df = pd.DataFrame(experience_records)
education_df = pd.DataFrame(education_records)
certificates_df = pd.DataFrame(certificate_records)
links_df = pd.DataFrame(link_records)

print("Raw extracted:")
print("  Pages:", len(pages_df))
print("  Skills:", len(skills_df))
print("  Projects:", len(projects_df))
print("  Experience:", len(experience_df))
print("  Education:", len(education_df))
print("  Certificate images:", len(certificates_df))
print("  Links:", len(links_df))


Raw extracted:
  Pages: 1
  Skills: 91
  Projects: 4
  Experience: 0
  Education: 3
  Certificate images: 28
  Links: 5


# ROLE / IDENTITY

In [ ]:
main_page = pages[0] if pages else {
    "soup": BeautifulSoup("", "lxml"),
    "text": "",
    "title": "",
    "url": portfolio_url
}

name = extract_name(
    main_page["soup"],
    main_page.get("title", "")
)

role_goal = extract_role_goal(
    main_page["soup"],
    main_page.get("text", ""),
    main_page.get("title", "")
)

candidate_profile = {
    "candidate_id": candidate_id,
    "portfolio_url": portfolio_url,
    "portfolio_domain": portfolio_domain,
    "name": name,
    "role_goal": role_goal,
    "scraped_at": datetime.now().isoformat()
}

candidate_profile


{'candidate_id': 'CAND_0001',
 'portfolio_url': 'https://varshh-hub.github.io/VARSHA---portfolio',
 'portfolio_domain': 'varshh-hub.github.io',
 'name': 'Turning Data into Insights.',
 'role_goal': 'Data Scientist',
 'scraped_at': '2026-08-21T13:56:27.511253'}

# LOGICAL DEDUPLICATION

In [ ]:
def dedupe_df(df, subset):
    if df is None or df.empty:
        return df

    usable = [
        column for column in subset
        if column in df.columns
    ]

    if not usable:
        return df.drop_duplicates()

    result = df.copy()

    for column in usable:
        result[column] = result[column].fillna("").astype(str).str.strip()

    return result.drop_duplicates(
        subset=usable,
        keep="first"
    ).reset_index(drop=True)


skills_df = dedupe_df(
    skills_df,
    ["candidate_id", "skill", "source_url", "evidence_text"]
)

projects_df = dedupe_df(
    projects_df,
    ["candidate_id", "project_name", "project_url"]
)

experience_df = dedupe_df(
    experience_df,
    ["candidate_id", "role", "company", "source_url"]
)

education_df = dedupe_df(
    education_df,
    ["candidate_id", "qualification", "institution", "year"]
)

# Certificate images: image hash is the strongest duplicate key.
certificates_df = dedupe_df(
    certificates_df,
    ["candidate_id", "image_sha256"]
)

links_df = dedupe_df(
    links_df,
    ["candidate_id", "link_url", "link_type"]
)

pages_df = dedupe_df(
    pages_df,
    ["candidate_id", "source_url"]
)

print("After logical deduplication:")
print("  Skills:", len(skills_df))
print("  Projects:", len(projects_df))
print("  Experience:", len(experience_df))
print("  Education:", len(education_df))
print("  Certificate images:", len(certificates_df))


After logical deduplication:
  Skills: 71
  Projects: 4
  Experience: 0
  Education: 3
  Certificate images: 27


# CERTIFICATE SUMMARY

In [ ]:
if not certificates_df.empty:
    certificate_summary = certificates_df[
        [
            "candidate_id",
            "certificate_name",
            "issuer",
            "date",
            "image_url",
            "local_image_path",
            "image_sha256",
            "ocr_text",
            "source_url",
            "error"
        ]
    ].copy()

    display(certificate_summary)
else:
    certificate_summary = pd.DataFrame()

    print(
        "No certificate images were detected. "
        "If certificates are loaded dynamically by JavaScript, "
        "use the Selenium fallback cell below."
    )


,candidate_id,certificate_name,issuer,date,image_url,local_image_path,image_sha256,ocr_text,source_url,error
0,CAND_0001,Linked [fJ Learning,,2026,https://varshh-hub.github.io/VARSHA---portfoli...,portfolio_data/certificate_images/23919e12853f...,23919e12853fc9be978953306d80f9bf147221f7e5b447...,Linked [fJ Learning\n\nPrompt Engineering Skil...,https://varshh-hub.github.io/VARSHA---portfolio,
1,CAND_0001,Linked [fJ Learning,,2026,https://varshh-hub.github.io/VARSHA---portfoli...,portfolio_data/certificate_images/5bb85e6475bf...,5bb85e6475bf935a1b8f9cad613ee3806f2273d8ef3d0b...,Linked [fJ Learning\n\nAl and Developer Produc...,https://varshh-hub.github.io/VARSHA---portfolio,
2,CAND_0001,Linked [fJ Learning,,2026,https://varshh-hub.github.io/VARSHA---portfoli...,portfolio_data/certificate_images/e13b2c790b3b...,e13b2c790b3b5c01bfb2617b97bf9a1addd82c8e105462...,Linked [fJ Learning\n\nIntroduction to Prompt ...,https://varshh-hub.github.io/VARSHA---portfolio,
3,CAND_0001,Linked [fJ Learning,,2026,https://varshh-hub.github.io/VARSHA---portfoli...,portfolio_data/certificate_images/d0ce80af9a66...,d0ce80af9a665f46f6f8f2da46fce83057fcd0a0943c6b...,Linked [fJ Learning\n\nPrompt Engineering with...,https://varshh-hub.github.io/VARSHA---portfolio,
4,CAND_0001,Linked [fJ Learning,,2026,https://varshh-hub.github.io/VARSHA---portfoli...,portfolio_data/certificate_images/3bee217e8c5a...,3bee217e8c5a5780987fc5abdd673f702396cbdc0c983b...,Linked [fJ Learning\n\nChatGPT: Perfecting You...,https://varshh-hub.github.io/VARSHA---portfolio,
5,CAND_0001,Linked [fJ Learning,,2026,https://varshh-hub.github.io/VARSHA---portfoli...,portfolio_data/certificate_images/d08c630e9dff...,d08c630e9dff2f357f0b51b45d9a0391162fb7db451e07...,Linked [fJ Learning\n\nBuilding Advanced Al Ap...,https://varshh-hub.github.io/VARSHA---portfolio,
6,CAND_0001,Linked [ff Learning,,2024,https://varshh-hub.github.io/VARSHA---portfoli...,portfolio_data/certificate_images/a3b887817e49...,a3b887817e49dd41b1c41ee17c31295140816a972584c4...,Linked [ff Learning\n\nStatistics Foundations ...,https://varshh-hub.github.io/VARSHA---portfolio,
7,CAND_0001,Linked [J Learning,,2024,https://varshh-hub.github.io/VARSHA---portfoli...,portfolio_data/certificate_images/e643ee71401f...,e643ee71401f56437e6442371ad60581ddd80313b8e75d...,Linked [J Learning\n\nStatistics Foundations 4...,https://varshh-hub.github.io/VARSHA---portfolio,
8,CAND_0001,Linked [J Learning,,2024,https://varshh-hub.github.io/VARSHA---portfoli...,portfolio_data/certificate_images/72bbc368b413...,72bbc368b413de2ece528a56647bf395c7af703ebfde18...,Linked [J Learning\n\nFundamentals to Become a...,https://varshh-hub.github.io/VARSHA---portfolio,
9,CAND_0001,Linked [J Learning,,2024,https://varshh-hub.github.io/VARSHA---portfoli...,portfolio_data/certificate_images/6cd8be79509e...,6cd8be79509e05b62a7fa1b8ac02dc715b430c8966c65a...,Linked [J Learning\n\nArtificial Intelligence ...,https://varshh-hub.github.io/VARSHA---portfolio,


# CANDIDATE SUMMARY


In [ ]:
candidate_df = pd.DataFrame([{
    **candidate_profile,
    "pages_found": len(pages),
    "external_links_found": len(crawl_result["external_links"]),
    "documents_found": len(crawl_result["document_links"]),
    "emails_found": len(crawl_result["email_links"]),
    "phones_found": len(crawl_result["phone_links"]),
    "skills_found": (
        skills_df["skill"].nunique()
        if not skills_df.empty and "skill" in skills_df.columns
        else 0
    ),
    "projects_found": len(projects_df),
    "experience_records": len(experience_df),
    "education_records": len(education_df),
    "certificate_images_found": len(certificates_df)
}])

display(candidate_df)


,candidate_id,portfolio_url,portfolio_domain,name,role_goal,scraped_at,pages_found,external_links_found,documents_found,emails_found,phones_found,skills_found,projects_found,experience_records,education_records,certificate_images_found
0,CAND_0001,https://varshh-hub.github.io/VARSHA---portfolio,varshh-hub.github.io,Turning Data into Insights.,Data Scientist,2026-08-21T13:56:27.511253,1,2,1,1,1,18,4,0,3,27


In [ ]:
def selenium_render(url):
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from webdriver_manager.chrome import ChromeDriverManager
    from selenium.webdriver.chrome.service import Service

    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1440,1200")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    try:
        driver.get(url)
        time.sleep(3)
        return driver.page_source
    finally:
        driver.quit()


In [ ]:
def upsert_candidate_table(df, filename, candidate_id):
    path = os.path.join(DATA_DIR, filename)

    if df is None:
        return

    df = df.copy()

    if "candidate_id" not in df.columns:
        df["candidate_id"] = candidate_id

    if os.path.exists(path):
        try:
            existing = pd.read_csv(path)
        except Exception:
            existing = pd.DataFrame()

        if not existing.empty and "candidate_id" in existing.columns:
            existing = existing[
                existing["candidate_id"].astype(str) != str(candidate_id)
            ]

        result = pd.concat(
            [existing, df],
            ignore_index=True,
            sort=False
        )
    else:
        result = df

    result.to_csv(
        path,
        index=False
    )

    print(f"Saved {len(df)} current records -> {path}")


def save_all_candidate_data():
    upsert_candidate_table(
        candidate_df,
        "candidates.csv",
        candidate_id
    )

    upsert_candidate_table(
        pages_df,
        "pages.csv",
        candidate_id
    )

    upsert_candidate_table(
        links_df,
        "links.csv",
        candidate_id
    )

    upsert_candidate_table(
        skills_df,
        "skills_evidence.csv",
        candidate_id
    )

    upsert_candidate_table(
        projects_df,
        "projects.csv",
        candidate_id
    )

    upsert_candidate_table(
        experience_df,
        "experience.csv",
        candidate_id
    )

    upsert_candidate_table(
        education_df,
        "education.csv",
        candidate_id
    )

    upsert_candidate_table(
        certificates_df,
        "certificates.csv",
        candidate_id
    )

    print("\nPortfolioIQ data saved successfully.")


save_all_candidate_data()


Saved 1 current records -> portfolio_data/candidates.csv
Saved 1 current records -> portfolio_data/pages.csv
Saved 5 current records -> portfolio_data/links.csv
Saved 71 current records -> portfolio_data/skills_evidence.csv
Saved 4 current records -> portfolio_data/projects.csv
Saved 0 current records -> portfolio_data/experience.csv
Saved 3 current records -> portfolio_data/education.csv
Saved 27 current records -> portfolio_data/certificates.csv

PortfolioIQ data saved successfully.


# FINAL REPORT

In [ ]:
print("=" * 70)
print("PORTFOLIOIQ SCRAPING COMPLETE")
print("=" * 70)
print("Candidate ID:", candidate_id)
print("Name:", candidate_profile.get("name", ""))
print("Role / Goal:", candidate_profile.get("role_goal", ""))
print()
print("Pages:", len(pages_df))
print("Skills:", skills_df["skill"].nunique() if not skills_df.empty else 0)
print("Projects:", len(projects_df))
print("Experience:", len(experience_df))
print("Education:", len(education_df))
print("Certificate images:", len(certificates_df))
print("External links:", len(crawl_result["external_links"]))
print()
print("Saved under:", os.path.abspath(DATA_DIR))
print("=" * 70)


PORTFOLIOIQ SCRAPING COMPLETE
Candidate ID: CAND_0001
Name: Turning Data into Insights.
Role / Goal: Data Scientist

Pages: 1
Skills: 18
Projects: 4
Experience: 0
Education: 3
Certificate images: 27
External links: 2

Saved under: /Users/varshh06/Desktop/1. PORTFOLIO INTELLIGENCE/portfolio_data


## What the verification stage can use

The scraper intentionally preserves evidence rather than marking claims as verified.

For example, if the candidate claims **Power BI**:

1. `skills_evidence.csv` can show where Power BI was mentioned.
2. `projects.csv` can show whether a project uses Power BI.
3. `links.csv` can show GitHub/LinkedIn/Kaggle/project links.
4. `certificates.csv` can show certificate image URLs and OCR text.
5. The verification layer can then decide whether the evidence is strong enough.

For certificate images specifically, the scraper keeps:

- original image URL
- local downloaded copy
- SHA-256 image hash
- OCR text
- extracted certificate name
- possible issuer
- possible year
- source portfolio URL

This means **30 certificate images can remain 30 separate evidence records** instead of being reduced to five keyword matches.
